# HuberRidgeAIME Scientific Reports revision reproducibility notebook

```bash
HRA_OUTPUT_DIR=./output      # override output location
HRA_QUICK_TEST=1             # smoke test; uses a small configuration
HRA_FORCE_RECOMPUTE=1        # recompute experiments instead of using cached CSV outputs
```

In [ ]:

# %% [revision setup]
# This addendum is self-contained and does not require result.zip from the original notebook.

import os, sys, io, json, math, time, zipfile, platform, warnings, urllib.request, hashlib, textwrap
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.special import softmax

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, log_loss
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV

warnings.filterwarnings("ignore")

try:
    from lightgbm import LGBMClassifier
    LGBM_AVAILABLE = True
except Exception:
    LGBM_AVAILABLE = False

# ------------------------
# Reproducibility switches
# ------------------------
REVISION_SEED = 42
np.random.seed(REVISION_SEED)

# Set HRA_QUICK_TEST=1 in the environment to do a tiny smoke test.
QUICK_TEST = bool(int(os.environ.get("HRA_QUICK_TEST", "0")))

PROJECT_ROOT = Path.cwd()
REVISION_OUTDIR = Path(os.environ.get("HRA_OUTPUT_DIR", PROJECT_ROOT / "output")).resolve()
REVISION_FIGDIR = REVISION_OUTDIR / "figures"
REVISION_TABLEDIR = REVISION_OUTDIR / "tables"
REVISION_DATADIR = REVISION_OUTDIR / "data"
REVISION_LOGDIR = REVISION_OUTDIR / "logs"
for p in [REVISION_OUTDIR, REVISION_FIGDIR, REVISION_TABLEDIR, REVISION_DATADIR, REVISION_LOGDIR]:
    p.mkdir(parents=True, exist_ok=True)

FORCE_RECOMPUTE = bool(int(os.environ.get("HRA_FORCE_RECOMPUTE", "0")))

REVISION_DATASETS = ["breast_cancer", "credit_approval", "har"]
REVISION_LEARNERS = ["lgbm", "svm", "mlp"]
REVISION_REPEATS = 3
MAX_N_PER_DATASET = {"breast_cancer": None, "credit_approval": None, "har": 3000}
MAX_CLONES = 100
BOOTSTRAPS = 12
NOISE_REPEATS = 8
DECOY_TRIALS = 12

if QUICK_TEST:
    REVISION_DATASETS = ["breast_cancer"]
    REVISION_LEARNERS = ["svm"]
    REVISION_REPEATS = 1
    MAX_N_PER_DATASET = {"breast_cancer": 300}
    MAX_CLONES = 20
    BOOTSTRAPS = 3
    NOISE_REPEATS = 2
    DECOY_TRIALS = 3

AIME_METHODS = ["AIME", "HuberAIME", "RidgeAIME", "HuberRidgeAIME"]

# Full factorial real-data pathology grid.
# clone_frac=0.50 means adding up to 50% cloned/linear-combination features, capped by MAX_CLONES.
OUTLIER_LEVELS = [0.00, 0.05, 0.10]
CLONE_FRACS = [0.00, 0.25, 0.50]
if QUICK_TEST:
    OUTLIER_LEVELS = [0.00, 0.10]
    CLONE_FRACS = [0.00, 0.50]

HUBER_DELTA = 1.0
RIDGE_LAMBDA = 1e-2
TOP_K = 10

print("Revision output directory:", REVISION_OUTDIR)
print("Force recompute:", FORCE_RECOMPUTE)
print("Datasets:", REVISION_DATASETS)
print("Learners:", REVISION_LEARNERS)
print("Repeats:", REVISION_REPEATS)
print("LightGBM available:", LGBM_AVAILABLE)


In [ ]:

# %% [helpers: formatting, saving, and reproducibility manifest]

def _safe_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan


def fmt_mean_std(mean, std, digits=3):
    if pd.isna(mean):
        return "--"
    if pd.isna(std):
        return f"{mean:.{digits}f}"
    return f"{mean:.{digits}f} $\\pm$ {std:.{digits}f}"


def fmt_p(p):
    if pd.isna(p):
        return "--"
    if p < 1e-4:
        return "$<10^{-4}$"
    return f"{p:.4f}"


def save_latex_table(df, path, caption=None, label=None, float_format="%.3f", index=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    latex = df.to_latex(index=index, escape=False, float_format=float_format)
    if caption or label:
        lines = ["\\begin{table}[t]", "\\centering"]
        lines.append(latex)
        if caption:
            lines.append(f"\\caption{{{caption}}}")
        if label:
            lines.append(f"\\label{{{label}}}")
        lines.append("\\end{table}")
        latex = "\n".join(lines)
    path.write_text(latex, encoding="utf-8")
    print("[tex]", path)
    return path


def save_csv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print("[csv]", path)
    return path


def file_sha256(path, block_size=2**20):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            b = f.read(block_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def write_environment_manifest():
    rows = []
    for mod in ["numpy", "pandas", "scipy", "sklearn", "matplotlib"]:
        try:
            m = __import__(mod)
            rows.append({"package": mod, "version": getattr(m, "__version__", "unknown")})
        except Exception:
            rows.append({"package": mod, "version": "not available"})
    rows.append({"package": "python", "version": sys.version.replace("\n", " ")})
    rows.append({"package": "platform", "version": platform.platform()})
    rows.append({"package": "lightgbm", "version": "available" if LGBM_AVAILABLE else "fallback used"})
    df = pd.DataFrame(rows)
    save_csv(df, REVISION_DATADIR / "environment_manifest.csv")
    return df

ENV_MANIFEST = write_environment_manifest()
ENV_MANIFEST


In [ ]:

# %% [data loading with missingness/provenance]

def _download_bytes(urls, timeout=180, retries=3, sleep=2):
    if isinstance(urls, str):
        urls = [urls]
    last = None
    for url in urls:
        for _ in range(retries):
            try:
                with urllib.request.urlopen(url, timeout=timeout) as r:
                    return r.read()
            except Exception as e:
                last = e
                time.sleep(sleep)
    raise last if last is not None else RuntimeError("download failed")


def make_unique(names):
    seen = {}
    out = []
    for name in names:
        name = str(name)
        if name not in seen:
            seen[name] = 0
            out.append(name)
        else:
            seen[name] += 1
            out.append(f"{name}__dup{seen[name]}")
    return out


def load_breast_cancer_dataset_revision():
    data = load_breast_cancer()
    X_raw = pd.DataFrame(data.data, columns=make_unique(data.feature_names))
    y = pd.Series(data.target.astype(int), name="target")
    meta = {
        "dataset": "breast_cancer",
        "description": "Breast Cancer Wisconsin Diagnostic; continuous nuclear features; binary classification.",
        "source": "sklearn.datasets.load_breast_cancer; originally UCI ML Repository",
        "access_link": "https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic",
        "license_note": "Public benchmark dataset; cite UCI/sklearn source as appropriate.",
        "raw_missing_cells": int(X_raw.isna().sum().sum()),
    }
    return X_raw, y, meta


def load_credit_raw_revision():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/credit-screening/crx.data"
    raw = pd.read_csv(url, header=None, na_values="?")
    raw.columns = [f"A{i+1}" for i in range(raw.shape[1]-1)] + ["target"]
    return raw


def process_credit_revision(protocol="impute"):
    raw = load_credit_raw_revision()
    if protocol == "complete_case":
        raw_proc = raw.dropna(axis=0).copy()
    elif protocol == "impute":
        raw_proc = raw.copy()
    else:
        raise ValueError(protocol)
    y = raw_proc["target"].map({"+": 1, "-": 0}).astype(int)
    X_df = raw_proc.drop(columns=["target"]).copy()
    cat_cols = [c for c in X_df.columns if X_df[c].dtype == object]
    num_cols = [c for c in X_df.columns if c not in cat_cols]
    for c in num_cols:
        X_df[c] = pd.to_numeric(X_df[c], errors="coerce")
        if protocol == "impute":
            X_df[c] = X_df[c].fillna(X_df[c].median())
    for c in cat_cols:
        X_df[c] = X_df[c].astype("object")
        if protocol == "impute":
            X_df[c] = X_df[c].fillna("Unknown")
    X_proc = pd.get_dummies(X_df, drop_first=True)
    X_proc.columns = make_unique([str(c) for c in X_proc.columns])
    return X_proc.astype(float), y.astype(int), raw


def load_credit_approval_revision():
    X_proc, y, raw = process_credit_revision(protocol="impute")
    meta = {
        "dataset": "credit_approval",
        "description": "Australian Credit Approval; mixed continuous/categorical variables; binary classification; missing values coded as '?'.",
        "source": "UCI ML Repository credit-screening/crx.data",
        "access_link": "https://archive.ics.uci.edu/dataset/143/statlog+australian+credit+approval",
        "license_note": "Public benchmark dataset; original feature names anonymized.",
        "raw_missing_cells": int(raw.drop(columns=["target"]).isna().sum().sum()),
    }
    return X_proc, y.reset_index(drop=True), meta


def load_har_revision():
    mirrors = [
        "https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip",
        "http://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip",
    ]
    by = _download_bytes(mirrors)
    zf = zipfile.ZipFile(io.BytesIO(by))
    root = "UCI HAR Dataset/"
    with zf.open(root + "features.txt") as f:
        feats = pd.read_csv(f, sep=r"\s+", header=None, names=["idx", "name"])
    feat_names = make_unique(feats["name"].astype(str).tolist())
    with zf.open(root + "train/X_train.txt") as f:
        Xtr = np.loadtxt(f).astype(float)
    with zf.open(root + "test/X_test.txt") as f:
        Xte = np.loadtxt(f).astype(float)
    with zf.open(root + "train/y_train.txt") as f:
        ytr = np.loadtxt(f).astype(int).ravel()
    with zf.open(root + "test/y_test.txt") as f:
        yte = np.loadtxt(f).astype(int).ravel()
    X = pd.DataFrame(np.vstack([Xtr, Xte]), columns=feat_names)
    y = pd.Series(np.hstack([ytr, yte]).astype(int) - 1, name="target")
    meta = {
        "dataset": "har",
        "description": "Human Activity Recognition Using Smartphones; sensor-derived continuous features; six-class classification.",
        "source": "UCI ML Repository UCI HAR Dataset.zip",
        "access_link": "https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones",
        "license_note": "Public benchmark dataset; original train/test split combined for stress experiments.",
        "raw_missing_cells": int(X.isna().sum().sum()),
    }
    return X, y, meta


def load_dataset_revision(name):
    if name == "breast_cancer":
        return load_breast_cancer_dataset_revision()
    if name == "credit_approval":
        return load_credit_approval_revision()
    if name == "har":
        return load_har_revision()
    raise ValueError(name)


def stratified_cap(X, y, max_n=None, seed=42):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=int)
    if max_n is None or len(y) <= max_n:
        return X, y
    splitter = StratifiedShuffleSplit(n_splits=1, train_size=max_n, random_state=seed)
    idx, _ = next(splitter.split(X, y))
    return X[idx], y[idx]


def standardize_array(X):
    return StandardScaler().fit_transform(np.asarray(X, dtype=float))


In [ ]:

# %% [dataset validity, missingness, outlier and multicollinearity diagnostics]

def effective_rank_from_singular_values(svals):
    svals = np.asarray(svals, dtype=float)
    svals = svals[svals > 1e-12]
    if svals.size == 0:
        return 0.0
    p = svals / svals.sum()
    return float(np.exp(-(p * np.log(p + 1e-15)).sum()))


def dataset_diagnostics(X):
    X = np.asarray(X, dtype=float)
    Xs = standardize_array(X)
    n, d = Xs.shape
    # condition number of X'X, using singular values for numerical stability
    svals = np.linalg.svd(Xs, full_matrices=False, compute_uv=False)
    cond_x = float((svals[0] / max(svals[-1], 1e-12)) ** 2) if svals.size else np.nan
    eff_rank = effective_rank_from_singular_values(svals)
    # pairwise correlation can be heavy for high-d; use a deterministic subset if needed
    if d > 700:
        cols = np.linspace(0, d-1, 700).astype(int)
        Xcorr = Xs[:, cols]
    else:
        Xcorr = Xs
    C = np.corrcoef(Xcorr, rowvar=False)
    C = np.nan_to_num(C, nan=0.0, posinf=0.0, neginf=0.0)
    off = np.abs(C[np.triu_indices_from(C, k=1)])
    max_abs_corr = float(np.max(off)) if off.size else 0.0
    mean_abs_corr = float(np.mean(off)) if off.size else 0.0
    frac_high_corr = float(np.mean(off > 0.90)) if off.size else 0.0
    med = np.nanmedian(Xs, axis=0)
    mad = np.nanmedian(np.abs(Xs - med), axis=0)
    mad = np.where(mad < 1e-12, 1.0, mad)
    robust_z = 0.6745 * (Xs - med) / mad
    outlier_cell_rate = float(np.mean(np.abs(robust_z) > 3.5))
    outlier_row_rate = float(np.mean(np.any(np.abs(robust_z) > 3.5, axis=1)))
    return {
        "n": n,
        "d_after_encoding": d,
        "condition_XtX": cond_x,
        "effective_rank": eff_rank,
        "effective_rank_ratio": eff_rank / max(d, 1),
        "max_abs_correlation": max_abs_corr,
        "mean_abs_correlation": mean_abs_corr,
        "frac_abs_corr_gt_0.90": frac_high_corr,
        "robust_outlier_cell_rate": outlier_cell_rate,
        "robust_outlier_row_rate": outlier_row_rate,
    }


def build_dataset_audit_table():
    audit_csv = REVISION_DATADIR / "revision_dataset_audit.csv"
    if audit_csv.exists() and not FORCE_RECOMPUTE:
        print("[cache] loading", audit_csv)
        df = pd.read_csv(audit_csv)
    else:
        rows = []
        for name in REVISION_DATASETS:
            X_df, y, meta = load_dataset_revision(name)
            diag = dataset_diagnostics(X_df.to_numpy(dtype=float))
            cls_counts = pd.Series(y).value_counts().sort_index().to_dict()
            rows.append({
                **meta,
                **diag,
                "n_classes": int(pd.Series(y).nunique()),
                "class_counts": "; ".join([f"{k}:{v}" for k, v in cls_counts.items()]),
                "missing_cell_rate_before_processing": meta["raw_missing_cells"] / max(1, X_df.shape[0] * X_df.shape[1]),
            })
        df = pd.DataFrame(rows)
        save_csv(df, audit_csv)

    tex_cols = [
        "dataset", "n", "d_after_encoding", "n_classes", "raw_missing_cells",
        "condition_XtX", "effective_rank_ratio", "max_abs_correlation",
        "frac_abs_corr_gt_0.90", "robust_outlier_cell_rate"
    ]
    tex_df = df[tex_cols].copy()
    rename = {
        "dataset": "Dataset",
        "n": "$n$",
        "d_after_encoding": "$d$",
        "n_classes": "Classes",
        "raw_missing_cells": "Missing cells",
        "condition_XtX": "$\\kappa(X^\\top X)$",
        "effective_rank_ratio": "Eff. rank / $d$",
        "max_abs_correlation": "Max |corr|",
        "frac_abs_corr_gt_0.90": "Frac |corr|>0.90",
        "robust_outlier_cell_rate": "Robust outlier cell rate",
    }
    tex_df = tex_df.rename(columns=rename)
    save_latex_table(
        tex_df,
        REVISION_TABLEDIR / "table_revision_dataset_validity_missingness.tex",
        caption=("Dataset validity, missingness, outlier diagnostics, and multicollinearity diagnostics. "
                 "The condition number and correlation summaries are computed after numeric encoding and standardization."),
        label="tab:revision_dataset_audit",
        float_format="%.3g",
    )
    return df

DATASET_AUDIT = build_dataset_audit_table()
DATASET_AUDIT


In [ ]:

# %% [AIME-family implementation with conditioning diagnostics]

def huber_weights_revision(r, delta=1.0):
    r = np.asarray(r, dtype=float)
    a = np.abs(r)
    w = np.ones_like(a)
    mask = a > delta
    w[mask] = delta / np.maximum(a[mask], 1e-12)
    return np.clip(w, 1e-8, None)


def _solve_weighted_ridge(X, y, w, lam):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    w = np.asarray(w, dtype=float)
    Xw = X * w[:, None]
    G = Xw.T @ X
    b = Xw.T @ y
    if lam > 0:
        G_reg = G + lam * np.eye(X.shape[1])
    else:
        G_reg = G
    try:
        coef = np.linalg.solve(G_reg, b)
    except np.linalg.LinAlgError:
        coef = np.linalg.pinv(G_reg) @ b
    return coef, G, G_reg


def _cond_and_eigs(G):
    G = np.asarray(G, dtype=float)
    try:
        evals = np.linalg.eigvalsh((G + G.T) / 2.0)
        evals = np.asarray(evals, dtype=float)
        lam_min = float(np.min(evals))
        lam_max = float(np.max(evals))
        cond = float(lam_max / max(lam_min, 1e-12))
        return cond, lam_min, lam_max
    except Exception:
        return np.nan, np.nan, np.nan


def compute_cwfi_aime_diag(X, Y, method="AIME", delta=1.0, ridge_lambda=1e-2, max_iter=50, tol=1e-6):
    """Return feature-importance matrix V and conditioning diagnostics.

    This implementation follows the code used in the reproducibility notebook:
    each class-probability column is regressed on the standardized feature matrix.
    Ridge regularization therefore directly stabilizes the ill-conditioned feature design.
    """
    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    n, d = X.shape
    C = Y.shape[1]

    mu = X.mean(axis=0, keepdims=True)
    sd = X.std(axis=0, keepdims=True)
    sd = np.where(sd < 1e-12, 1.0, sd)
    Xs = (X - mu) / sd

    use_huber = method.lower().startswith("huber")
    use_ridge = "ridge" in method.lower()
    lam = float(ridge_lambda if use_ridge else 0.0)

    V = np.zeros((d, C), dtype=float)
    diag_rows = []
    for c in range(C):
        y = Y[:, c]
        w = np.ones(n, dtype=float)
        coef = np.zeros(d, dtype=float)
        n_iter = 1
        if use_huber:
            for it in range(max_iter):
                prev = coef.copy()
                coef, G, G_reg = _solve_weighted_ridge(Xs, y, w, lam)
                residual = y - Xs @ coef
                w = huber_weights_revision(residual, delta=delta)
                n_iter = it + 1
                if np.linalg.norm(coef - prev) <= tol * (np.linalg.norm(prev) + 1e-12):
                    break
        else:
            coef, G, G_reg = _solve_weighted_ridge(Xs, y, w, lam)
        V[:, c] = coef / sd.ravel()
        cond_raw, min_raw, max_raw = _cond_and_eigs(G)
        cond_reg, min_reg, max_reg = _cond_and_eigs(G_reg)
        diag_rows.append({
            "class_index": c,
            "cond_raw": cond_raw,
            "cond_reg": cond_reg,
            "lambda_min_raw": min_raw,
            "lambda_min_reg": min_reg,
            "lambda_max_raw": max_raw,
            "lambda_max_reg": max_reg,
            "n_iter": n_iter,
            "mean_huber_weight": float(np.mean(w)),
            "frac_downweighted": float(np.mean(w < 0.999)),
            "coef_l2_norm": float(np.linalg.norm(coef)),
        })
    diag = pd.DataFrame(diag_rows)
    summary = {
        "cond_raw_median": float(diag["cond_raw"].median()),
        "cond_reg_median": float(diag["cond_reg"].median()),
        "lambda_min_raw_median": float(diag["lambda_min_raw"].median()),
        "lambda_min_reg_median": float(diag["lambda_min_reg"].median()),
        "mean_iter": float(diag["n_iter"].mean()),
        "mean_huber_weight": float(diag["mean_huber_weight"].mean()),
        "frac_downweighted": float(diag["frac_downweighted"].mean()),
        "coef_l2_norm_mean": float(diag["coef_l2_norm"].mean()),
    }
    return V, summary, diag


In [ ]:

# %% [metrics: full-vector stability, decoy mass, rank-sensitive diagnostics]

def global_strength(V):
    V = np.asarray(V, dtype=float)
    return np.abs(V).sum(axis=1)


def _topk_from_scores(scores, k=10):
    scores = np.asarray(scores, dtype=float)
    if scores.size == 0:
        return set()
    k = min(k, scores.size)
    return set(np.argpartition(-scores, k-1)[:k].tolist())


def topk_jaccard(a, b, k=10):
    A = _topk_from_scores(a, k)
    B = _topk_from_scores(b, k)
    if len(A | B) == 0:
        return np.nan
    return len(A & B) / len(A | B)


def spearman_safe(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if a.size < 2 or np.nanstd(a) < 1e-12 or np.nanstd(b) < 1e-12:
        return np.nan
    return float(stats.spearmanr(a, b, nan_policy="omit").correlation)


def cosine_safe(a, b):
    a = np.asarray(a, dtype=float).ravel()
    b = np.asarray(b, dtype=float).ravel()
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom < 1e-12:
        return np.nan
    return float(np.dot(a, b) / denom)


def relative_l2_similarity(a, b):
    a = np.asarray(a, dtype=float).ravel()
    b = np.asarray(b, dtype=float).ravel()
    denom = np.linalg.norm(a) + 1e-12
    return float(1.0 - np.linalg.norm(a - b) / denom)


def compare_attribution_matrices(V0, V1, k=10):
    g0 = global_strength(V0)
    g1 = global_strength(V1)
    return {
        "topk_jaccard": topk_jaccard(g0, g1, k=k),
        "spearman_global": spearman_safe(g0, g1),
        "cosine_flat": cosine_safe(V0, V1),
        "relative_l2_similarity": relative_l2_similarity(V0, V1),
    }


def make_decoy_augmented_X(X, n_decoys=3, seed=42):
    """Add irrelevant decoys that the black-box output does not depend on.

    Decoys include permuted, correlated, and interaction-like columns. Because Y is kept fixed,
    any attribution assigned to these columns is spurious.
    """
    rng = np.random.default_rng(seed)
    X = np.asarray(X, dtype=float)
    n, d = X.shape
    decoys = []
    names = []
    if n_decoys >= 1:
        j = int(rng.integers(0, d))
        decoys.append(rng.permutation(X[:, j]))
        names.append("permuted_feature_decoy")
    if n_decoys >= 2:
        j1, j2 = rng.choice(d, size=2, replace=True)
        noise = rng.normal(0, 0.02 * (np.std(X[:, j1]) + 1e-12), size=n)
        decoys.append(0.85 * X[:, j1] + 0.15 * X[:, j2] + noise)
        names.append("correlated_linear_decoy")
    if n_decoys >= 3:
        j1, j2 = rng.choice(d, size=2, replace=True)
        prod = X[:, j1] * X[:, j2]
        prod = (prod - np.mean(prod)) / (np.std(prod) + 1e-12)
        decoys.append(prod + rng.normal(0, 0.02, size=n))
        names.append("interaction_decoy")
    for q in range(3, n_decoys):
        j = int(rng.integers(0, d))
        decoys.append(rng.permutation(X[:, j]))
        names.append(f"extra_permuted_decoy_{q}")
    D = np.column_stack(decoys) if decoys else np.empty((n, 0))
    X_ext = np.column_stack([X, D])
    decoy_idx = list(range(d, d + D.shape[1]))
    return X_ext, decoy_idx, names


def decoy_metrics_from_V(V, decoy_idx, k=10):
    g = global_strength(V)
    total = float(np.sum(g) + 1e-12)
    decoy_mass = float(np.sum(g[decoy_idx]) / total) if decoy_idx else np.nan
    ranks = pd.Series(-g).rank(method="min").to_numpy()  # rank 1 = largest importance
    decoy_ranks = ranks[decoy_idx] if decoy_idx else np.array([np.nan])
    d_total = len(g)
    topk = _topk_from_scores(g, k)
    infiltration = float(sum(i in topk for i in decoy_idx) / max(1, len(decoy_idx)))
    best_rank = float(np.nanmin(decoy_ranks))
    # Percentile near 1 = decoy ranked very high; lower is better.
    best_rank_percentile = float(1.0 - (best_rank - 1.0) / max(1.0, d_total - 1.0))
    return {
        "decoy_mass_ratio": decoy_mass,
        "decoy_resistance_mass": 1.0 - decoy_mass,
        "decoy_topk_infiltration": infiltration,
        "decoy_best_rank": best_rank,
        "decoy_best_rank_percentile": best_rank_percentile,
    }


In [ ]:

# %% [pathology injection and learner fitting]

def inject_outliers_revision(X, rate=0.10, scale=8.0, feature_frac=0.15, seed=42):
    rng = np.random.default_rng(seed)
    X = np.asarray(X, dtype=float).copy()
    n, d = X.shape
    if rate <= 0:
        return X, {"n_outlier_rows": 0, "outlier_feature_frac": 0.0}
    m = max(1, int(round(rate * n)))
    rows = rng.choice(n, size=m, replace=False)
    q = max(1, int(round(feature_frac * d)))
    for i in rows:
        cols = rng.choice(d, size=q, replace=False)
        # Heavy-tailed perturbation; clipped to avoid numerical overflow.
        perturb = rng.standard_t(df=2, size=q) * scale
        perturb = np.clip(perturb, -5 * scale, 5 * scale)
        X[i, cols] += perturb
    return X, {"n_outlier_rows": int(m), "outlier_feature_frac": float(q / max(1, d))}


def inject_collinearity_revision(X, clone_frac=0.50, clone_noise=0.01, max_clones=100, seed=42):
    rng = np.random.default_rng(seed)
    X = np.asarray(X, dtype=float)
    n, d = X.shape
    if clone_frac <= 0:
        return X, {"n_clones": 0, "clone_noise": np.nan}
    q = int(round(clone_frac * d))
    q = max(1, min(q, max_clones))
    source = rng.choice(d, size=q, replace=True)
    clones = []
    for s in source:
        noise = rng.normal(0, clone_noise * (np.std(X[:, s]) + 1e-12), size=n)
        # A near-duplicate plus a small contribution from another feature.
        t = int(rng.integers(0, d))
        clones.append(0.95 * X[:, s] + 0.05 * X[:, t] + noise)
    X_aug = np.column_stack([X, np.column_stack(clones)])
    return X_aug, {"n_clones": int(q), "clone_noise": float(clone_noise)}


def apply_pathology_revision(X, outlier_rate=0.0, clone_frac=0.0, seed=42):
    X0 = np.asarray(X, dtype=float)
    X1, info_out = inject_outliers_revision(X0, rate=outlier_rate, seed=seed)
    X2, info_col = inject_collinearity_revision(X1, clone_frac=clone_frac, max_clones=MAX_CLONES, seed=seed + 1000)
    info = {**info_out, **info_col, "outlier_rate": outlier_rate, "clone_frac": clone_frac, "d_before": X0.shape[1], "d_after": X2.shape[1]}
    return X2, info


def make_revision_learner(kind, seed=42):
    if kind == "lgbm":
        if LGBM_AVAILABLE:
            return LGBMClassifier(
                n_estimators=250, learning_rate=0.05, max_depth=10,
                subsample=0.9, colsample_bytree=0.9, random_state=seed,
                verbosity=-1, n_jobs=-1,
            )
        # Fallback keeps the notebook executable without LightGBM.
        return HistGradientBoostingClassifier(max_iter=250, learning_rate=0.05, random_state=seed)
    if kind == "svm":
        return make_pipeline(StandardScaler(), SVC(C=10.0, kernel="rbf", gamma="scale", probability=True, random_state=seed))
    if kind == "mlp":
        return make_pipeline(
            StandardScaler(),
            MLPClassifier(hidden_layer_sizes=(100, 50), activation="relu", learning_rate_init=1e-3,
                          alpha=1e-4, max_iter=250, early_stopping=True, random_state=seed)
        )
    raise ValueError(kind)


def fit_predictor_revision(X, y, learner="svm", seed=42):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=seed, stratify=y
    )
    model = make_revision_learner(learner, seed=seed)
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    fit_time = time.perf_counter() - t0
    Y_test = model.predict_proba(X_test)
    y_pred = np.argmax(Y_test, axis=1)
    # Map class argmax to true class labels if needed.
    classes = getattr(model, "classes_", None)
    if classes is None and hasattr(model, "named_steps"):
        classes = getattr(list(model.named_steps.values())[-1], "classes_", None)
    if classes is not None:
        y_pred_label = np.asarray(classes)[y_pred]
    else:
        y_pred_label = y_pred
    acc = accuracy_score(y_test, y_pred_label)
    try:
        ll = log_loss(y_test, Y_test, labels=np.unique(y))
    except Exception:
        ll = np.nan
    return model, X_train, X_test, y_train, y_test, Y_test, {"fit_time_sec": fit_time, "test_accuracy": acc, "test_log_loss": ll}


In [ ]:

# %% [stress metrics for one method/model condition]

def bootstrap_attr_stability(X, Y, method, V0=None, B=12, seed=42):
    rng = np.random.default_rng(seed)
    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    n = X.shape[0]
    if V0 is None:
        V0, _, _ = compute_cwfi_aime_diag(X, Y, method=method, delta=HUBER_DELTA, ridge_lambda=RIDGE_LAMBDA)
    vals = []
    for _ in range(B):
        idx = rng.choice(n, size=n, replace=True)
        Vb, _, _ = compute_cwfi_aime_diag(X[idx], Y[idx], method=method, delta=HUBER_DELTA, ridge_lambda=RIDGE_LAMBDA)
        vals.append(compare_attribution_matrices(V0, Vb, k=TOP_K))
    df = pd.DataFrame(vals)
    return {f"bootstrap_{c}": float(df[c].mean()) for c in df.columns}


def noise_attr_stability(X, model, method, V0=None, R=8, noise_sd=0.02, seed=42):
    rng = np.random.default_rng(seed)
    X = np.asarray(X, dtype=float)
    if V0 is None:
        Y0 = model.predict_proba(X)
        V0, _, _ = compute_cwfi_aime_diag(X, Y0, method=method, delta=HUBER_DELTA, ridge_lambda=RIDGE_LAMBDA)
    vals = []
    col_sd = np.std(X, axis=0, keepdims=True)
    col_sd = np.where(col_sd < 1e-12, 1.0, col_sd)
    for _ in range(R):
        Xn = X + rng.normal(0, noise_sd, size=X.shape) * col_sd
        Yn = model.predict_proba(Xn)
        Vn, _, _ = compute_cwfi_aime_diag(Xn, Yn, method=method, delta=HUBER_DELTA, ridge_lambda=RIDGE_LAMBDA)
        vals.append(compare_attribution_matrices(V0, Vn, k=TOP_K))
    df = pd.DataFrame(vals)
    return {f"noise_{c}": float(df[c].mean()) for c in df.columns}


def decoy_attr_metrics(X, Y, method, trials=12, seed=42):
    rng = np.random.default_rng(seed)
    vals = []
    for t in range(trials):
        X_ext, decoy_idx, _ = make_decoy_augmented_X(X, n_decoys=3, seed=int(rng.integers(0, 2**31-1)))
        Vd, _, _ = compute_cwfi_aime_diag(X_ext, Y, method=method, delta=HUBER_DELTA, ridge_lambda=RIDGE_LAMBDA)
        vals.append(decoy_metrics_from_V(Vd, decoy_idx, k=TOP_K))
    df = pd.DataFrame(vals)
    return {f"decoy_{c}": float(df[c].mean()) for c in df.columns}


In [ ]:

# %% [main controlled real-data pathology experiment]

def run_revision_pathology_experiments(force=FORCE_RECOMPUTE):
    raw_path = REVISION_DATADIR / "revision_pathology_results_raw.csv"
    if raw_path.exists() and not force:
        print("[cache] loading", raw_path)
        return pd.read_csv(raw_path)

    rows = []
    total_jobs = len(REVISION_DATASETS) * len(REVISION_LEARNERS) * REVISION_REPEATS * len(OUTLIER_LEVELS) * len(CLONE_FRACS)
    job = 0

    for dataset in REVISION_DATASETS:
        X_df, y_series, meta = load_dataset_revision(dataset)
        X_base = standardize_array(X_df.to_numpy(dtype=float))
        y_base = y_series.to_numpy(dtype=int)
        X_base, y_base = stratified_cap(X_base, y_base, max_n=MAX_N_PER_DATASET.get(dataset), seed=REVISION_SEED)
        for rep in range(REVISION_REPEATS):
            for outlier_rate in OUTLIER_LEVELS:
                for clone_frac in CLONE_FRACS:
                    seed = REVISION_SEED + 10000 * rep + int(1000 * outlier_rate) + int(100 * clone_frac)
                    X_path, pinfo = apply_pathology_revision(X_base, outlier_rate=outlier_rate, clone_frac=clone_frac, seed=seed)
                    condition_name = f"out{outlier_rate:.2f}_clone{clone_frac:.2f}"
                    for learner in REVISION_LEARNERS:
                        job += 1
                        print(f"[{job}/{total_jobs}] dataset={dataset} learner={learner} rep={rep} condition={condition_name} X={X_path.shape}")
                        try:
                            model, Xtr, Xte, ytr, yte, Yte, model_info = fit_predictor_revision(X_path, y_base, learner=learner, seed=seed)
                        except Exception as e:
                            rows.append({
                                "dataset": dataset, "learner": learner, "repeat": rep,
                                "outlier_rate": outlier_rate, "clone_frac": clone_frac,
                                "condition": condition_name, "method": "MODEL_FIT_FAILED", "error": str(e),
                            })
                            continue

                        for method in AIME_METHODS:
                            t0 = time.perf_counter()
                            try:
                                V0, diag_summary, _ = compute_cwfi_aime_diag(
                                    Xte, Yte, method=method, delta=HUBER_DELTA, ridge_lambda=RIDGE_LAMBDA
                                )
                                explain_time = time.perf_counter() - t0
                                boot = bootstrap_attr_stability(Xte, Yte, method, V0=V0, B=BOOTSTRAPS, seed=seed + 1)
                                noise = noise_attr_stability(Xte, model, method, V0=V0, R=NOISE_REPEATS, noise_sd=0.02, seed=seed + 2)
                                decoy = decoy_attr_metrics(Xte, Yte, method, trials=DECOY_TRIALS, seed=seed + 3)
                                row = {
                                    "dataset": dataset,
                                    "learner": learner,
                                    "repeat": rep,
                                    "outlier_rate": outlier_rate,
                                    "clone_frac": clone_frac,
                                    "condition": condition_name,
                                    "method": method,
                                    "n_total": int(X_path.shape[0]),
                                    "n_test": int(Xte.shape[0]),
                                    "d_before": int(pinfo["d_before"]),
                                    "d_after": int(pinfo["d_after"]),
                                    "n_outlier_rows": int(pinfo["n_outlier_rows"]),
                                    "n_clones": int(pinfo["n_clones"]),
                                    **model_info,
                                    "explain_time_sec": float(explain_time),
                                    **diag_summary,
                                    **boot,
                                    **noise,
                                    **decoy,
                                    "error": "",
                                }
                                rows.append(row)
                            except Exception as e:
                                rows.append({
                                    "dataset": dataset, "learner": learner, "repeat": rep,
                                    "outlier_rate": outlier_rate, "clone_frac": clone_frac,
                                    "condition": condition_name, "method": method,
                                    "error": str(e), **pinfo, **model_info,
                                })

    raw = pd.DataFrame(rows)
    save_csv(raw, raw_path)
    return raw

REVISION_RAW = run_revision_pathology_experiments(force=FORCE_RECOMPUTE)
REVISION_RAW.head()


In [ ]:

# %% [summaries and paired tests]

def holm_adjust(pvals):
    p = np.asarray(pvals, dtype=float)
    m = len(p)
    order = np.argsort(p)
    adj = np.empty(m, dtype=float)
    running = 0.0
    for rank, idx in enumerate(order):
        val = (m - rank) * p[idx]
        running = max(running, val)
        adj[idx] = min(1.0, running)
    return adj


def hodges_lehmann(d):
    d = np.asarray(d, dtype=float)
    d = d[np.isfinite(d)]
    n = len(d)
    if n == 0:
        return np.nan
    vals = []
    for i in range(n):
        vals.extend(0.5 * (d[i] + d[i:]))
    return float(np.median(vals))


def paired_test_revision(df, baseline="HuberAIME", target="HuberRidgeAIME", metrics=None, subset_query=None):
    if metrics is None:
        metrics = [
            ("bootstrap_spearman_global", "greater"),
            ("bootstrap_cosine_flat", "greater"),
            ("noise_spearman_global", "greater"),
            ("noise_cosine_flat", "greater"),
            ("decoy_decoy_resistance_mass", "greater"),
            ("decoy_decoy_mass_ratio", "less"),
            ("cond_reg_median", "less"),
            ("coef_l2_norm_mean", "less"),
        ]
    work = df.copy()
    if "error" in work.columns:
        err_ok = work["error"].isna() | (work["error"].astype(str) == "")
    else:
        err_ok = pd.Series(True, index=work.index)
    work = work[work["method"].isin([baseline, target]) & err_ok]
    if subset_query:
        work = work.query(subset_query)
    key_cols = ["dataset", "learner", "repeat", "outlier_rate", "clone_frac", "condition"]
    rows = []
    for metric, direction in metrics:
        piv = work.pivot_table(index=key_cols, columns="method", values=metric, aggfunc="mean")
        if baseline not in piv.columns or target not in piv.columns:
            continue
        piv = piv[[baseline, target]].dropna()
        if piv.empty:
            continue
        if direction == "greater":
            diff = piv[target] - piv[baseline]
            alt = "greater"
        elif direction == "less":
            diff = piv[baseline] - piv[target]  # positive means target is smaller and therefore better
            alt = "greater"
        else:
            diff = piv[target] - piv[baseline]
            alt = "two-sided"
        diff = diff.replace([np.inf, -np.inf], np.nan).dropna()
        if len(diff) < 2 or np.allclose(diff, 0):
            stat, pval = np.nan, 1.0
        else:
            stat, pval = stats.wilcoxon(diff, alternative=alt, zero_method="wilcox")
        rows.append({
            "metric": metric,
            "better_direction_for_target": direction,
            "n_pairs": int(len(diff)),
            "mean_target": float(piv[target].mean()),
            "mean_baseline": float(piv[baseline].mean()),
            "mean_better_diff": float(diff.mean()) if len(diff) else np.nan,
            "HL_better_diff": hodges_lehmann(diff),
            "p_raw": float(pval),
        })
    out = pd.DataFrame(rows)
    if not out.empty:
        out["p_holm"] = holm_adjust(out["p_raw"].to_numpy())
    return out


def build_revision_summaries(raw):
    ok = raw[(raw["method"].isin(AIME_METHODS)) & (raw["error"].fillna("") == "")].copy()
    save_csv(ok, REVISION_DATADIR / "revision_pathology_results_clean.csv")

    summary_metrics = [
        "bootstrap_topk_jaccard", "bootstrap_spearman_global", "bootstrap_cosine_flat",
        "noise_topk_jaccard", "noise_spearman_global", "noise_cosine_flat",
        "decoy_decoy_mass_ratio", "decoy_decoy_resistance_mass", "decoy_decoy_topk_infiltration",
        "cond_raw_median", "cond_reg_median", "lambda_min_reg_median", "coef_l2_norm_mean",
        "explain_time_sec", "test_accuracy",
    ]
    summary = ok.groupby(["outlier_rate", "clone_frac", "method"], as_index=False)[summary_metrics].agg(["mean", "std"])
    summary.columns = ["_".join([c for c in col if c]) for col in summary.columns.to_flat_index()]
    summary = summary.reset_index()
    save_csv(summary, REVISION_DATADIR / "revision_pathology_summary_by_condition_method.csv")

    # Focus table: clean vs joint high-pathology condition; HRA and HuberAIME.
    focus = ok[((ok["outlier_rate"] == 0.0) & (ok["clone_frac"] == 0.0)) |
               ((ok["outlier_rate"] == max(OUTLIER_LEVELS)) & (ok["clone_frac"] == max(CLONE_FRACS)))]
    focus = focus[focus["method"].isin(["HuberAIME", "HuberRidgeAIME"])]
    focus_summary = focus.groupby(["condition", "method"], as_index=False).agg({
        "bootstrap_spearman_global": ["mean", "std"],
        "noise_cosine_flat": ["mean", "std"],
        "decoy_decoy_resistance_mass": ["mean", "std"],
        "cond_reg_median": ["median"],
        "coef_l2_norm_mean": ["mean", "std"],
        "explain_time_sec": ["mean", "std"],
    })
    focus_summary.columns = ["_".join([c for c in col if c]) for col in focus_summary.columns.to_flat_index()]
    tex_focus = focus_summary.copy()
    tex_focus["Bootstrap Spearman"] = [fmt_mean_std(m, s) for m, s in zip(tex_focus["bootstrap_spearman_global_mean"], tex_focus["bootstrap_spearman_global_std"])]
    tex_focus["Noise cosine"] = [fmt_mean_std(m, s) for m, s in zip(tex_focus["noise_cosine_flat_mean"], tex_focus["noise_cosine_flat_std"])]
    tex_focus["Decoy resistance mass"] = [fmt_mean_std(m, s) for m, s in zip(tex_focus["decoy_decoy_resistance_mass_mean"], tex_focus["decoy_decoy_resistance_mass_std"])]
    tex_focus["Median $\\kappa_{reg}$"] = tex_focus["cond_reg_median_median"].map(lambda x: f"{x:.2e}")
    tex_focus["Coef. norm"] = [fmt_mean_std(m, s) for m, s in zip(tex_focus["coef_l2_norm_mean_mean"], tex_focus["coef_l2_norm_mean_std"])]
    tex_focus["Runtime (s)"] = [fmt_mean_std(m, s) for m, s in zip(tex_focus["explain_time_sec_mean"], tex_focus["explain_time_sec_std"])]
    cond_col = "condition_" if "condition_" in tex_focus.columns else "condition"
    method_col = "method_" if "method_" in tex_focus.columns else "method"
    tex_focus = tex_focus[[cond_col, method_col, "Bootstrap Spearman", "Noise cosine", "Decoy resistance mass", "Median $\\kappa_{reg}$", "Coef. norm", "Runtime (s)"]]
    tex_focus = tex_focus.rename(columns={cond_col: "Condition", method_col: "Method"})
    save_latex_table(
        tex_focus,
        REVISION_TABLEDIR / "table_revision_hra_vs_huberaime_pathology.tex",
        caption=("HuberAIME versus HuberRidgeAIME on clean and joint outlier+collinearity real-data stress conditions. "
                 "The table reports full-vector metrics added to avoid Top-K saturation/floor effects."),
        label="tab:revision_hra_huberaime_pathology",
    )

    paired_all = paired_test_revision(ok, baseline="HuberAIME", target="HuberRidgeAIME")
    save_csv(paired_all, REVISION_DATADIR / "revision_paired_tests_hra_vs_huberaime_all.csv")
    paired_high = paired_test_revision(
        ok, baseline="HuberAIME", target="HuberRidgeAIME",
        subset_query=f"outlier_rate == {max(OUTLIER_LEVELS)} and clone_frac == {max(CLONE_FRACS)}"
    )
    save_csv(paired_high, REVISION_DATADIR / "revision_paired_tests_hra_vs_huberaime_joint_high.csv")

    for name, tbl in [("all", paired_all), ("joint_high", paired_high)]:
        tex = tbl.copy()
        if not tex.empty:
            tex["p_raw"] = tex["p_raw"].map(fmt_p)
            tex["p_holm"] = tex["p_holm"].map(fmt_p)
            for c in ["mean_target", "mean_baseline", "mean_better_diff", "HL_better_diff"]:
                tex[c] = tex[c].map(lambda x: f"{x:.4g}" if pd.notna(x) else "--")
        save_latex_table(
            tex,
            REVISION_TABLEDIR / f"table_revision_paired_tests_hra_vs_huberaime_{name}.tex",
            caption=("Paired Wilcoxon tests comparing HuberRidgeAIME with HuberAIME. "
                     "For metrics marked 'less', the reported difference is baseline minus HRA, so positive values favor HRA."),
            label=f"tab:revision_paired_hra_huberaime_{name}",
        )
    return ok, summary, paired_all, paired_high

REVISION_CLEAN, REVISION_SUMMARY, REVISION_PAIRED_ALL, REVISION_PAIRED_HIGH = build_revision_summaries(REVISION_RAW)
REVISION_PAIRED_HIGH


In [ ]:

# %% [figures: schematic, pathology heatmap, conditioning, metric discrimination]

def fig_schematic_hra_problem(out_path=REVISION_FIGDIR / "fig_revision_aime_hra_schematic.png"):
    fig, ax = plt.subplots(figsize=(13.5, 6.2), dpi=220)
    ax.axis("off")
    ax.set_xlim(0, 12)
    ax.set_ylim(0, 6)

    def box(x, y, w, h, text, fontsize=10):
        rect = plt.Rectangle((x, y), w, h, fill=False, linewidth=1.5)
        ax.add_patch(rect)
        ax.text(x + w/2, y + h/2, text, ha="center", va="center", fontsize=fontsize, wrap=True)
        return rect

    box(0.4, 4.2, 2.0, 0.8, "Input data\n$X$", 11)
    box(3.0, 4.2, 2.1, 0.8, "Black-box model\n$f(X)=Y$", 11)
    box(5.8, 4.2, 2.7, 0.8, "Single AIME-family\nexplanation map", 11)
    box(9.2, 4.2, 2.3, 0.8, "Global/local\nattributions", 11)

    for x0, x1, y in [(2.4, 3.0, 4.6), (5.1, 5.8, 4.6), (8.5, 9.2, 4.6)]:
        ax.annotate("", xy=(x1, y), xytext=(x0, y), arrowprops=dict(arrowstyle="->", lw=1.5))

    ax.text(0.4, 3.65, "Least-squares AIME", fontsize=12, weight="bold")
    box(0.4, 2.35, 3.2, 0.9, "$\\min_A \\|Y - XA\\|_F^2$", 10)
    ax.text(4.0, 3.65, "Problem under stress", fontsize=12, weight="bold")
    box(4.0, 2.05, 3.45, 1.25, "Outliers create large residuals\nCollinearity makes $X^\\top W X$ ill-conditioned\nTop-K metrics can saturate or hit floor", 9)
    ax.text(8.1, 3.65, "HuberRidgeAIME", fontsize=12, weight="bold")
    box(8.1, 2.05, 3.6, 1.25, "$\\sum_i \\rho_\\delta(r_i)+\\frac{\\lambda}{2}\\|A\\|_F^2$\nHuber weights downweight outliers\nRidge lifts small eigenvalues", 9)

    ax.annotate("", xy=(8.1, 2.65), xytext=(7.45, 2.65), arrowprops=dict(arrowstyle="->", lw=1.5))
    ax.text(0.5, 1.15,
            "Revision experiments: standard public datasets are retained, but controlled outlier and collinearity stressors are injected\n"
            "to align the empirical evaluation with the stated robustness motivation. Full-vector and decoy-mass metrics supplement Top-K Jaccard.",
            fontsize=10, va="top")
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    print("[png]", out_path)
    return out_path


def fig_pathology_heatmap(clean_df, metric="bootstrap_spearman_global", out_path=REVISION_FIGDIR / "fig_revision_pathology_heatmap_hra_minus_huberaime.png"):
    work = clean_df[clean_df["method"].isin(["HuberAIME", "HuberRidgeAIME"])].copy()
    keys = ["dataset", "learner", "repeat", "outlier_rate", "clone_frac"]
    piv = work.pivot_table(index=keys, columns="method", values=metric, aggfunc="mean").reset_index()
    piv = piv.dropna(subset=["HuberAIME", "HuberRidgeAIME"])
    piv["diff"] = piv["HuberRidgeAIME"] - piv["HuberAIME"]
    grid = piv.groupby(["outlier_rate", "clone_frac"], as_index=False)["diff"].mean()
    mat = grid.pivot(index="outlier_rate", columns="clone_frac", values="diff").sort_index(ascending=True)

    fig, ax = plt.subplots(figsize=(5.2, 4.2), dpi=220)
    im = ax.imshow(mat.values, aspect="auto", origin="lower")
    ax.set_xticks(np.arange(len(mat.columns)))
    ax.set_yticks(np.arange(len(mat.index)))
    ax.set_xticklabels([f"{c:.2f}" for c in mat.columns])
    ax.set_yticklabels([f"{r:.2f}" for r in mat.index])
    ax.set_xlabel("Clone fraction / collinearity stress")
    ax.set_ylabel("Outlier rate")
    ax.set_title("HRA $-$ HuberAIME: full-vector bootstrap Spearman")
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax.text(j, i, f"{mat.values[i,j]:.3f}", ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax, shrink=0.85, label="Mean difference")
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    print("[png]", out_path)
    return out_path


def fig_conditioning_boxplot(clean_df, out_path=REVISION_FIGDIR / "fig_revision_conditioning_boxplot.png"):
    work = clean_df[clean_df["method"].isin(["HuberAIME", "HuberRidgeAIME"])].copy()
    high = work[(work["outlier_rate"] == max(OUTLIER_LEVELS)) & (work["clone_frac"] == max(CLONE_FRACS))]
    if high.empty:
        high = work
    fig, ax = plt.subplots(figsize=(6.2, 4.0), dpi=220)
    labels = []
    data = []
    for method in ["HuberAIME", "HuberRidgeAIME"]:
        vals = high.loc[high["method"] == method, "cond_reg_median"].replace([np.inf, -np.inf], np.nan).dropna()
        data.append(np.log10(vals + 1e-12))
        labels.append(method)
    ax.boxplot(data, labels=labels, showfliers=False)
    ax.set_ylabel("$\\log_{10}$ regularized condition number")
    ax.set_title("Ridge term stabilizes the weighted normal matrix\n(joint high outlier+collinearity condition)")
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    print("[png]", out_path)
    return out_path


def fig_metric_discrimination(clean_df, out_path=REVISION_FIGDIR / "fig_revision_metric_discrimination_decoy.png"):
    work = clean_df.copy()
    high = work[(work["outlier_rate"] == max(OUTLIER_LEVELS)) & (work["clone_frac"] == max(CLONE_FRACS))]
    if high.empty:
        high = work
    methods = [m for m in AIME_METHODS if m in high["method"].unique()]
    fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.8), dpi=220)
    data1 = [high.loc[high["method"] == m, "decoy_decoy_topk_infiltration"].dropna() for m in methods]
    data2 = [high.loc[high["method"] == m, "decoy_decoy_mass_ratio"].dropna() for m in methods]
    axes[0].boxplot(data1, labels=methods, showfliers=False)
    axes[0].set_title("Top-K decoy infiltration\n(can saturate)")
    axes[0].set_ylabel("Lower is better")
    axes[0].tick_params(axis="x", rotation=30)
    axes[1].boxplot(data2, labels=methods, showfliers=False)
    axes[1].set_title("Decoy importance mass\n(non-saturating)")
    axes[1].tick_params(axis="x", rotation=30)
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    print("[png]", out_path)
    return out_path

SCHEMATIC_PATH = fig_schematic_hra_problem()
HEATMAP_PATH = fig_pathology_heatmap(REVISION_CLEAN)
COND_PATH = fig_conditioning_boxplot(REVISION_CLEAN)
METRIC_PATH = fig_metric_discrimination(REVISION_CLEAN)


In [ ]:

# %% [missing-data comparison]

def evaluate_credit_missing_protocol(protocol="impute", learner="svm", seed=42):
    X_df, y, raw = process_credit_revision(protocol=protocol)
    X = standardize_array(X_df.to_numpy(dtype=float))
    y = y.to_numpy(dtype=int)
    model, Xtr, Xte, ytr, yte, Yte, info = fit_predictor_revision(X, y, learner=learner, seed=seed)
    rows = []
    for method in AIME_METHODS:
        V, diag, _ = compute_cwfi_aime_diag(Xte, Yte, method=method, delta=HUBER_DELTA, ridge_lambda=RIDGE_LAMBDA)
        boot = bootstrap_attr_stability(Xte, Yte, method, V0=V, B=max(3, BOOTSTRAPS//2), seed=seed+1)
        decoy = decoy_attr_metrics(Xte, Yte, method, trials=max(3, DECOY_TRIALS//2), seed=seed+2)
        rows.append({
            "dataset": "credit_approval",
            "protocol": protocol,
            "learner": learner,
            "n_after_protocol": int(X.shape[0]),
            "d_after_protocol": int(X.shape[1]),
            "raw_missing_cells": int(raw.drop(columns=["target"]).isna().sum().sum()),
            "method": method,
            **info,
            **diag,
            **boot,
            **decoy,
        })
    return rows


def run_missing_data_comparison(force=FORCE_RECOMPUTE):
    out = REVISION_DATADIR / "revision_missing_data_comparison_raw.csv"
    if out.exists() and not force:
        return pd.read_csv(out)
    if QUICK_TEST and os.environ.get("HRA_RUN_CREDIT_MISSING_IN_QUICK", "0") != "1":
        df = pd.DataFrame([{
            "dataset": "credit_approval", "protocol": "skipped_in_quick_test",
            "learner": "--", "method": "FAILED",
            "error": "Skipped because HRA_QUICK_TEST=1. Set HRA_RUN_CREDIT_MISSING_IN_QUICK=1 to run."
        }])
        save_csv(df, out)
        return df
    rows = []
    for learner in REVISION_LEARNERS:
        for protocol in ["impute", "complete_case"]:
            try:
                rows.extend(evaluate_credit_missing_protocol(protocol=protocol, learner=learner, seed=REVISION_SEED))
            except Exception as e:
                rows.append({"dataset": "credit_approval", "protocol": protocol, "learner": learner, "method": "FAILED", "error": str(e)})
    df = pd.DataFrame(rows)
    save_csv(df, out)
    return df

MISSING_COMP_RAW = run_missing_data_comparison(force=FORCE_RECOMPUTE)
miss_ok = MISSING_COMP_RAW[MISSING_COMP_RAW["method"].isin(AIME_METHODS)].copy() if "method" in MISSING_COMP_RAW.columns else pd.DataFrame()

if miss_ok.empty:
    # This can happen in an offline smoke test because the credit dataset is downloaded from UCI.
    miss_sum = pd.DataFrame(columns=[
        "protocol", "learner", "method", "n_after_protocol", "d_after_protocol",
        "test_accuracy", "bootstrap_spearman_global", "decoy_decoy_resistance_mass", "cond_reg_median"
    ])
    note = pd.DataFrame([{
        "Missing-data protocol": "not executed",
        "Learner": "--",
        "Method": "--",
        "$n$": "--",
        "$d$": "--",
        "Accuracy": "--",
        "Bootstrap Spearman": "--",
        "Decoy resistance mass": "--",
        "Median $\\kappa_{reg}$": "--",
    }])
    save_csv(miss_sum, REVISION_DATADIR / "revision_missing_data_comparison_summary.csv")
    save_latex_table(
        note,
        REVISION_TABLEDIR / "table_revision_missing_data_comparison.tex",
        caption=("Missing-data handling comparison on the Australian Credit Approval dataset. "
                 "This table is populated when the UCI credit dataset is accessible."),
        label="tab:revision_missing_comparison",
    )
else:
    miss_sum = miss_ok.groupby(["protocol", "learner", "method"], as_index=False).agg({
        "n_after_protocol": "first",
        "d_after_protocol": "first",
        "test_accuracy": "mean",
        "bootstrap_spearman_global": "mean",
        "decoy_decoy_resistance_mass": "mean",
        "cond_reg_median": "median",
    })
    save_csv(miss_sum, REVISION_DATADIR / "revision_missing_data_comparison_summary.csv")
    tex = miss_sum.copy()
    for c in ["test_accuracy", "bootstrap_spearman_global", "decoy_decoy_resistance_mass"]:
        tex[c] = tex[c].map(lambda x: f"{x:.3f}" if pd.notna(x) else "--")
    tex["cond_reg_median"] = tex["cond_reg_median"].map(lambda x: f"{x:.2e}" if pd.notna(x) else "--")
    tex = tex.rename(columns={
        "protocol": "Missing-data protocol",
        "learner": "Learner",
        "method": "Method",
        "n_after_protocol": "$n$",
        "d_after_protocol": "$d$",
        "test_accuracy": "Accuracy",
        "bootstrap_spearman_global": "Bootstrap Spearman",
        "decoy_decoy_resistance_mass": "Decoy resistance mass",
        "cond_reg_median": "Median $\\kappa_{reg}$",
    })
    save_latex_table(
        tex,
        REVISION_TABLEDIR / "table_revision_missing_data_comparison.tex",
        caption=("Missing-data handling comparison on the Australian Credit Approval dataset. "
                 "The imputed protocol uses median/mode-style processing with an Unknown category, whereas complete-case removes rows with missing values."),
        label="tab:revision_missing_comparison",
    )
miss_sum.head()


In [ ]:

# %% [runtime scaling benchmark for computational-cost discussion]

def generate_synthetic_runtime_data(n=1000, d=100, C=3, rho=0.7, seed=42):
    rng = np.random.default_rng(seed)
    idx = np.arange(d)
    cov = rho ** np.abs(np.subtract.outer(idx, idx))
    # Use Cholesky for moderate d; fallback to independent if numerical issue.
    try:
        L = np.linalg.cholesky(cov + 1e-8 * np.eye(d))
        X = rng.normal(size=(n, d)) @ L.T
    except Exception:
        X = rng.normal(size=(n, d))
    B = rng.normal(size=(d, C)) / np.sqrt(max(1, d))
    logits = X @ B + 0.25 * rng.normal(size=(n, C))
    Y = softmax(logits, axis=1)
    # Add mild contamination to make Huber iterations non-trivial.
    X, _ = inject_outliers_revision(X, rate=0.05, scale=6.0, seed=seed + 1)
    return X, Y


def run_runtime_scaling(force=FORCE_RECOMPUTE):
    out = REVISION_DATADIR / "revision_runtime_scaling_raw.csv"
    if out.exists() and not force:
        return pd.read_csv(out)
    n_values = [500, 1000, 2000, 4000]
    d_values = [20, 100, 300]
    if QUICK_TEST:
        n_values = [200, 500]
        d_values = [20, 80]
    rows = []
    for n in n_values:
        for d in d_values:
            for rep in range(3 if not QUICK_TEST else 1):
                X, Y = generate_synthetic_runtime_data(n=n, d=d, C=3, seed=REVISION_SEED + 1000*rep + n + d)
                for method in AIME_METHODS:
                    t0 = time.perf_counter()
                    V, diag, _ = compute_cwfi_aime_diag(X, Y, method=method, delta=HUBER_DELTA, ridge_lambda=RIDGE_LAMBDA)
                    elapsed = time.perf_counter() - t0
                    rows.append({
                        "n": n, "d": d, "C": Y.shape[1], "repeat": rep, "method": method,
                        "runtime_sec": elapsed, "n_times_d": n*d, **diag,
                    })
                    print(f"runtime n={n} d={d} method={method}: {elapsed:.3f}s")
    df = pd.DataFrame(rows)
    save_csv(df, out)
    return df

RUNTIME_RAW = run_runtime_scaling(force=FORCE_RECOMPUTE)

runtime_summary = RUNTIME_RAW.groupby(["method", "n", "d"], as_index=False).agg(runtime_mean=("runtime_sec", "mean"), runtime_std=("runtime_sec", "std"))
save_csv(runtime_summary, REVISION_DATADIR / "revision_runtime_scaling_summary.csv")

# Compact table by method over all sizes.
runtime_overall = RUNTIME_RAW.groupby("method", as_index=False).agg(
    mean_runtime_sec=("runtime_sec", "mean"),
    median_runtime_sec=("runtime_sec", "median"),
    max_runtime_sec=("runtime_sec", "max"),
    mean_iterations=("mean_iter", "mean"),
)
tex = runtime_overall.copy()
for c in ["mean_runtime_sec", "median_runtime_sec", "max_runtime_sec", "mean_iterations"]:
    tex[c] = tex[c].map(lambda x: f"{x:.3f}" if pd.notna(x) else "--")
tex = tex.rename(columns={
    "method": "Method",
    "mean_runtime_sec": "Mean runtime (s)",
    "median_runtime_sec": "Median runtime (s)",
    "max_runtime_sec": "Max runtime (s)",
    "mean_iterations": "Mean IRLS iterations",
})
save_latex_table(
    tex,
    REVISION_TABLEDIR / "table_revision_runtime_scaling.tex",
    caption=("Runtime scaling of AIME-family methods on controlled synthetic matrices. "
             "Huber variants require IRLS iterations, whereas ridge adds negligible overhead relative to solving the weighted normal equations."),
    label="tab:revision_runtime_scaling",
)

# Runtime figure.
fig, ax = plt.subplots(figsize=(6.2, 4.0), dpi=220)
for method, g in runtime_summary.groupby("method"):
    gg = g.groupby("n_times_d" if "n_times_d" in g.columns else ["n", "d"]).mean(numeric_only=True)
# Rebuild with n_times_d available.
runtime_plot = RUNTIME_RAW.groupby(["method", "n_times_d"], as_index=False)["runtime_sec"].mean()
for method, g in runtime_plot.groupby("method"):
    g = g.sort_values("n_times_d")
    ax.plot(g["n_times_d"], g["runtime_sec"], marker="o", label=method)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("$n \\times d$ (log scale)")
ax.set_ylabel("Runtime seconds (log scale)")
ax.set_title("AIME-family computational cost")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
runtime_fig_path = REVISION_FIGDIR / "fig_revision_runtime_scaling.png"
fig.savefig(runtime_fig_path, bbox_inches="tight")
plt.close(fig)
print("[png]", runtime_fig_path)
runtime_overall


In [ ]:

# %% [figure provenance, artifact manifest, and ZIP packaging]

def build_figure_provenance_table():
    rows = [
        {
            "artifact": "fig_revision_aime_hra_schematic.png",
            "type": "figure",
            "source_data": "Generated schematic; no third-party image used",
            "generating_function": "fig_schematic_hra_problem",
            "purpose": "",
        },
        {
            "artifact": "fig_revision_pathology_heatmap_hra_minus_huberaime.png",
            "type": "figure",
            "source_data": "revision_pathology_results_clean.csv",
            "generating_function": "fig_pathology_heatmap",
            "purpose": "",
        },
        {
            "artifact": "fig_revision_conditioning_boxplot.png",
            "type": "figure",
            "source_data": "revision_pathology_results_clean.csv",
            "generating_function": "fig_conditioning_boxplot",
            "purpose": "",
        },
        {
            "artifact": "fig_revision_metric_discrimination_decoy.png",
            "type": "figure",
            "source_data": "revision_pathology_results_clean.csv",
            "generating_function": "fig_metric_discrimination",
            "purpose": "",
        },
        {
            "artifact": "fig_revision_runtime_scaling.png",
            "type": "figure",
            "source_data": "revision_runtime_scaling_raw.csv",
            "generating_function": "run_runtime_scaling + plotting cell",
            "purpose": "",
        },
        {
            "artifact": "table_revision_dataset_validity_missingness.tex",
            "type": "table",
            "source_data": "revision_dataset_audit.csv",
            "generating_function": "build_dataset_audit_table",
            "purpose": "",
        },
        {
            "artifact": "table_revision_missing_data_comparison.tex",
            "type": "table",
            "source_data": "revision_missing_data_comparison_raw.csv",
            "generating_function": "run_missing_data_comparison",
            "purpose": "",
        },
        {
            "artifact": "table_revision_hra_vs_huberaime_pathology.tex",
            "type": "table",
            "source_data": "revision_pathology_results_clean.csv",
            "generating_function": "build_revision_summaries",
            "purpose": "",
        },
        {
            "artifact": "table_revision_paired_tests_hra_vs_huberaime_joint_high.tex",
            "type": "table",
            "source_data": "revision_paired_tests_hra_vs_huberaime_joint_high.csv",
            "generating_function": "paired_test_revision",
            "purpose": "",
        },
    ]
    df = pd.DataFrame(rows)
    save_csv(df, REVISION_DATADIR / "revision_figure_provenance.csv")
    save_latex_table(
        df,
        REVISION_TABLEDIR / "table_revision_figure_provenance.tex",
        caption=("Figure and table provenance for the revision. All figures were generated by the authors from the listed CSV files or as original schematics; no third-party images were used."),
        label="tab:revision_figure_provenance",
    )
    return df

PROVENANCE = build_figure_provenance_table()

# Artifact manifest with checksums for DOI archive.
manifest_rows = []
for p in sorted(REVISION_OUTDIR.rglob("*")):
    if p.is_file():
        manifest_rows.append({
            "relative_path": str(p.relative_to(REVISION_OUTDIR)),
            "bytes": p.stat().st_size,
            "sha256": file_sha256(p),
        })
ARTIFACT_MANIFEST = pd.DataFrame(manifest_rows)
save_csv(ARTIFACT_MANIFEST, REVISION_OUTDIR / "revision_artifact_manifest.csv")

zip_path = REVISION_OUTDIR / "HuberRidgeAIME_revision_outputs_for_zenodo.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(REVISION_OUTDIR.rglob("*")):
        if p.is_file() and p.resolve() != zip_path.resolve():
            zf.write(p, arcname=str(p.relative_to(REVISION_OUTDIR)))
print("[zip]", zip_path)

PROVENANCE


In [ ]:
# %% [final output tree]
# Print a compact final artifact tree. Everything should be under ./output.
from pathlib import Path

def print_output_tree(root=REVISION_OUTDIR, max_files_per_dir=40):
    root = Path(root)
    print(f"[final output root] {root.resolve()}")
    for sub in ["data", "tables", "figures", "logs"]:
        d = root / sub
        if not d.exists():
            continue
        files = sorted([p for p in d.iterdir() if p.is_file()])
        print(f"\n{sub}/ ({len(files)} files)")
        for p in files[:max_files_per_dir]:
            print(f"  - {p.name} ({p.stat().st_size:,} bytes)")
        if len(files) > max_files_per_dir:
            print(f"  ... {len(files)-max_files_per_dir} more files")

print_output_tree()


## Revision addendum: log-scaled conditioning and coefficient-norm outputs

This cell converts extremely large conditioning and coefficient-norm diagnostics to publication-friendly
log-scale summaries. It generates additional CSV, LaTeX, and PNG outputs for the manuscript and
supplementary information.


In [ ]:

# %% [log-scaled conditioning and coefficient-norm outputs]
# This addendum is designed to be appended after the revision experiments have produced:
#   data/revision_pathology_results_clean.csv
# It creates publication-friendly log10 summaries for condition numbers and coefficient norms.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import zipfile

# Fall back to standard output folders if the notebook variables are not available.
try:
    REVISION_OUTDIR
except NameError:
    REVISION_OUTDIR = Path.cwd() / "output"
    REVISION_DATADIR = REVISION_OUTDIR / "data"
    REVISION_TABLEDIR = REVISION_OUTDIR / "tables"
    REVISION_FIGDIR = REVISION_OUTDIR / "figures"

REVISION_DATADIR.mkdir(parents=True, exist_ok=True)
REVISION_TABLEDIR.mkdir(parents=True, exist_ok=True)
REVISION_FIGDIR.mkdir(parents=True, exist_ok=True)

def _safe_log10_positive(x):
    """Return log10(x) for positive finite x; otherwise NaN."""
    x = pd.to_numeric(x, errors="coerce")
    x = x.where(np.isfinite(x) & (x > 0), np.nan)
    return np.log10(x)

def _mean_sd_str(s, digits=2):
    s = pd.to_numeric(s, errors="coerce")
    m = s.mean()
    sd = s.std(ddof=1)
    if pd.isna(m):
        return "--"
    if pd.isna(sd):
        return f"{m:.{digits}f}"
    return f"{m:.{digits}f} $\\pm$ {sd:.{digits}f}"

def _fmt_num(x, digits=2):
    if pd.isna(x):
        return "--"
    return f"{x:.{digits}f}"

def _write_latex_table(df, path, caption, label):
    tex = df.to_latex(index=False, escape=False, caption=caption, label=label)
    Path(path).write_text(tex, encoding="utf-8")
    print("[saved]", path)

def build_log_conditioning_outputs(force=False):
    raw_path = REVISION_DATADIR / "revision_pathology_results_clean.csv"
    if not raw_path.exists():
        raw_path = REVISION_DATADIR / "revision_pathology_results_raw.csv"
    if not raw_path.exists():
        raise FileNotFoundError(
            "revision_pathology_results_clean.csv or revision_pathology_results_raw.csv was not found. "
            "Run the controlled real-data pathology experiment first."
        )

    df = pd.read_csv(raw_path)
    df = df[df.get("error").isna() if "error" in df.columns else np.ones(len(df), dtype=bool)].copy()

    needed = ["cond_raw_median", "cond_reg_median", "coef_l2_norm_mean"]
    for col in needed:
        if col not in df.columns:
            raise KeyError(f"Required column not found: {col}")

    df["log10_cond_raw"] = _safe_log10_positive(df["cond_raw_median"])
    df["log10_cond_reg"] = _safe_log10_positive(df["cond_reg_median"])
    df["log10_coef_l2_norm"] = _safe_log10_positive(df["coef_l2_norm_mean"])

    # Joint high-stress condition: maximum outlier rate and maximum clone fraction in the experiment.
    max_out = pd.to_numeric(df["outlier_rate"], errors="coerce").max()
    max_clone = pd.to_numeric(df["clone_frac"], errors="coerce").max()
    df["stress_group"] = np.where(
        (np.isclose(df["outlier_rate"], max_out)) & (np.isclose(df["clone_frac"], max_clone)),
        "joint high",
        "all conditions"
    )

    long_path = REVISION_DATADIR / "revision_log_conditioning_long.csv"
    df.to_csv(long_path, index=False)
    print("[saved]", long_path)

    # Summary by method over all controlled real-data pathology conditions.
    summary_rows = []
    for method, g in df.groupby("method", sort=False):
        summary_rows.append({
            "Method": method,
            "$\\log_{10}\\kappa(Y^\\top W Y + \\lambda I)$": _mean_sd_str(g["log10_cond_reg"]),
            "$\\log_{10}\\|A\\|_F$": _mean_sd_str(g["log10_coef_l2_norm"]),
            "Median IRLS iter.": _fmt_num(pd.to_numeric(g.get("mean_iter", np.nan), errors="coerce").median(), 1),
            "Mean explain time (s)": _fmt_num(pd.to_numeric(g.get("explain_time_sec", np.nan), errors="coerce").mean(), 4),
        })
    summary = pd.DataFrame(summary_rows)
    summary_csv = REVISION_DATADIR / "revision_log_conditioning_summary.csv"
    summary.to_csv(summary_csv, index=False)
    print("[saved]", summary_csv)

    _write_latex_table(
        summary,
        REVISION_TABLEDIR / "table_revision_log_conditioning_summary.tex",
        caption=(
            "Log-scaled numerical conditioning and coefficient-norm diagnostics across the controlled "
            "real-data pathology experiments. Values are mean $\\pm$ standard deviation unless otherwise noted."
        ),
        label="tab:revision_log_conditioning_summary",
    )

    # Paired HRA-vs-HuberAIME diagnostic table for all and joint high-stress conditions.
    pair_metrics = ["log10_cond_reg", "log10_coef_l2_norm", "bootstrap_cosine_flat", "noise_cosine_flat"]
    key_cols = ["dataset", "learner", "repeat", "outlier_rate", "clone_frac"]
    pair_rows = []
    for group_name, subset in [("All conditions", df), ("Joint high stress", df[df["stress_group"] == "joint high"])]:
        piv = subset.pivot_table(index=key_cols, columns="method", values=pair_metrics, aggfunc="mean")
        if ("HuberRidgeAIME" not in piv.columns.get_level_values(1)) or ("HuberAIME" not in piv.columns.get_level_values(1)):
            continue
        for metric in pair_metrics:
            hra = piv[(metric, "HuberRidgeAIME")]
            huber = piv[(metric, "HuberAIME")]
            diff = hra - huber
            # Negative is better for log10 condition number and log10 coefficient norm.
            if metric in ["log10_cond_reg", "log10_coef_l2_norm"]:
                better_text = "lower is better"
            else:
                better_text = "higher is better"
            pair_rows.append({
                "Stress subset": group_name,
                "Metric": metric.replace("_", "\\_"),
                "HRA mean": _fmt_num(hra.mean(), 3),
                "HuberAIME mean": _fmt_num(huber.mean(), 3),
                "HRA--HuberAIME": _fmt_num(diff.mean(), 3),
                "Median diff.": _fmt_num(diff.median(), 3),
                "Direction": better_text,
            })
    pair_df = pd.DataFrame(pair_rows)
    pair_csv = REVISION_DATADIR / "revision_log_conditioning_hra_vs_huberaime.csv"
    pair_df.to_csv(pair_csv, index=False)
    print("[saved]", pair_csv)

    _write_latex_table(
        pair_df,
        REVISION_TABLEDIR / "table_revision_log_conditioning_hra_vs_huberaime.tex",
        caption=(
            "Paired comparison between HuberRidgeAIME and HuberAIME using log-scaled conditioning diagnostics "
            "and non-saturating full-vector agreement metrics. Negative differences are favorable for "
            "$\\log_{10}\\kappa$ and $\\log_{10}\\|A\\|_F$."
        ),
        label="tab:revision_log_conditioning_hra_vs_huberaime",
    )

    # Separate figures to avoid overcrowding and to make each figure usable independently.
    # Figure A: log10 regularized condition number.
    fig_df = df[["method", "log10_cond_reg"]].dropna()
    methods = [m for m in ["AIME", "HuberAIME", "RidgeAIME", "HuberRidgeAIME"] if m in fig_df["method"].unique()]
    data = [fig_df.loc[fig_df["method"] == m, "log10_cond_reg"].to_numpy() for m in methods]
    fig, ax = plt.subplots(figsize=(7.5, 4.8), dpi=220)
    ax.boxplot(data, tick_labels=methods, showfliers=False)
    ax.set_ylabel(r"$\log_{10}\kappa(Y^\top W Y + \lambda I)$")
    ax.set_title("Log-scaled conditioning under controlled real-data stress")
    ax.tick_params(axis="x", rotation=25)
    fig.tight_layout()
    out_fig1 = REVISION_FIGDIR / "fig_revision_log_condition_number_boxplot.png"
    fig.savefig(out_fig1, bbox_inches="tight")
    plt.close(fig)
    print("[saved]", out_fig1)

    # Figure B: log10 coefficient norm.
    fig_df = df[["method", "log10_coef_l2_norm"]].dropna()
    data = [fig_df.loc[fig_df["method"] == m, "log10_coef_l2_norm"].to_numpy() for m in methods]
    fig, ax = plt.subplots(figsize=(7.5, 4.8), dpi=220)
    ax.boxplot(data, tick_labels=methods, showfliers=False)
    ax.set_ylabel(r"$\log_{10}\|A\|_F$")
    ax.set_title("Log-scaled coefficient norm under controlled real-data stress")
    ax.tick_params(axis="x", rotation=25)
    fig.tight_layout()
    out_fig2 = REVISION_FIGDIR / "fig_revision_log_coef_norm_boxplot.png"
    fig.savefig(out_fig2, bbox_inches="tight")
    plt.close(fig)
    print("[saved]", out_fig2)

    # Update figure provenance, if it exists.
    prov_path = REVISION_DATADIR / "revision_figure_provenance.csv"
    new_prov = pd.DataFrame([
        {
            "artifact": "fig_revision_log_condition_number_boxplot.png",
            "type": "figure",
            "source_data": "revision_log_conditioning_long.csv",
            "generating_function": "build_log_conditioning_outputs",
            "purpose": "",
        },
        {
            "artifact": "fig_revision_log_coef_norm_boxplot.png",
            "type": "figure",
            "source_data": "revision_log_conditioning_long.csv",
            "generating_function": "build_log_conditioning_outputs",
            "purpose": "",
        },
        {
            "artifact": "table_revision_log_conditioning_summary.tex",
            "type": "table",
            "source_data": "revision_log_conditioning_summary.csv",
            "generating_function": "build_log_conditioning_outputs",
            "purpose": "Main/Supplementary table: publication-friendly log10 conditioning and coefficient norm summary",
        },
        {
            "artifact": "table_revision_log_conditioning_hra_vs_huberaime.tex",
            "type": "table",
            "source_data": "revision_log_conditioning_hra_vs_huberaime.csv",
            "generating_function": "build_log_conditioning_outputs",
            "purpose": "",
        },
    ])
    if prov_path.exists():
        prov = pd.read_csv(prov_path)
        prov = pd.concat([prov, new_prov], ignore_index=True)
        prov = prov.drop_duplicates(subset=["artifact"], keep="last")
    else:
        prov = new_prov
    prov.to_csv(prov_path, index=False)
    print("[updated]", prov_path)

    # Update artifact manifest, if it exists.
    manifest_path = REVISION_OUTDIR / "revision_artifact_manifest.csv"
    artifacts = []
    for subdir in ["data", "tables", "figures"]:
        for p in sorted((REVISION_OUTDIR / subdir).glob("*")):
            if p.is_file():
                artifacts.append({"relative_path": str(p.relative_to(REVISION_OUTDIR)), "bytes": p.stat().st_size})
    pd.DataFrame(artifacts).to_csv(manifest_path, index=False)
    print("[updated]", manifest_path)

    # Rebuild Zenodo package to include the new log-scaled outputs.
    zip_path = REVISION_OUTDIR / "HuberRidgeAIME_revision_outputs_for_zenodo.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in REVISION_OUTDIR.rglob("*"):
            if p.is_file() and p.resolve() != zip_path.resolve():
                zf.write(p, p.relative_to(REVISION_OUTDIR))
    print("[updated zip]", zip_path)

    return df, summary, pair_df

log_conditioning_long, log_conditioning_summary, log_conditioning_hra_vs_huberaime = build_log_conditioning_outputs(force=True)
display(log_conditioning_summary)
display(log_conditioning_hra_vs_huberaime)
